# organic_ratio — entry notebook


## 1. Git sync

Репозиторий `organic_ratio` пока не создан в гите — ячейка ниже закомментирована. Включить, когда заведут репу (поправить `ORG/REPO/BRANCH/USER`).

In [3]:
import os
import subprocess
import base64
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = "/home/jovyan/KEDRO/organic_ratio"
BRANCH = "main"
ORG = "azur-games"
REPO = "organic_ratio"
USER = "Anton-Filimoncev-azur"

os.chdir(PROJECT_ROOT)
print("PWD:", Path().resolve())

load_dotenv()
TOKEN = os.getenv("GIT_PAT")
if not TOKEN:
    raise ValueError("GIT_PAT not found in environment")

auth = base64.b64encode(f"{USER}:{TOKEN}".encode()).decode()

subprocess.run(["git", "rebase", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "merge", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "reset", "--hard"], check=True)
subprocess.run(["git", "clean", "-fd"], check=True)

subprocess.run(
    [
        "git",
        "-c",
        f"http.extraheader=Authorization: Basic {auth}",
        "fetch",
        f"https://github.com/{ORG}/{REPO}.git",
        BRANCH,
    ],
    check=True,
)
subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], check=True)

print("Repository synced successfully.")

PWD: /home/jovyan/KEDRO/organic_ratio


fatal: not a git repository (or any parent up to mount point /home)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


CalledProcessError: Command '['git', 'reset', '--hard']' returned non-zero exit status 128.

## 2. Env

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## 3. Ingestion (S3 → data/raw/partition/)

In [ ]:
%run run.py

## 4. Per-source preprocessing (raw → data/features/partition/)

In [ ]:
import gc
%run run_preprocessing.py
gc.collect()

## 5. Cohort aggregation (TODO)

`run_cohort_aggregation.py` — агрегирует user-grain features в cohort-grain по ключам `country_code × media_source × platform × install_date`.

In [ ]:
# %run run_cohort_aggregation.py

## 6. Train / Eval / Inference (TODO)

PyMC иерархическая Beta-регрессия для cohort-level organic share.

In [ ]:
# %run run_train.py
# %run run_eval.py
# %run run_inference.py

## (опц.) Пакетный запуск экспериментов через CONFIG_OVERRIDE_PATH

In [ ]:
# import os
# from pathlib import Path
# from omegaconf import OmegaConf
#
# SWEEP_PATH = Path("conf/batch_training/sweep.yml")
# TMP_DIR = Path("conf/batch_training/.tmp")
#
# sweep_cfg = OmegaConf.load(SWEEP_PATH)
# TMP_DIR.mkdir(parents=True, exist_ok=True)
#
# for exp in sweep_cfg.experiments:
#     run_name = exp.name
#     override_path = TMP_DIR / f"{run_name}.yml"
#     OmegaConf.save(config=exp.overrides, f=override_path)
#     os.environ["CONFIG_OVERRIDE_PATH"] = str(override_path)
#     try:
#         %run run_train.py
#         %run run_eval.py
#     finally:
#         os.environ.pop("CONFIG_OVERRIDE_PATH", None)